# TGT-M: the full modeling pipeline (Review 2)
**Team 7 · SRM IST · 21CSC314P Big Data Essentials.** *TGT-M: A Temporal Gated Transformer Model for Seasonal Groundwater Stress Assessment and Cross-Region Generalization Across India Using Multimodal Data.*

This notebook shows **how the whole modeling study is organised and run**, retrains a small part of it live, and prints the result tables.

| Part | What it does |
|---|---|
| 1 | Finds the project files |
| 2 | Checks the data (253,828 well-rounds, 84 columns, leakage-safe model inputs) |
| 3 | **Code map**: every script in `modeling/`, what it does, and the order they run in |
| 4 | Runs the models: **`quick`** (XGBoost + ANN, 10-20 min), **`full`** (everything, hours) or **`view`** (nothing) |
| 5 | The result tables: main comparison, rainfall effect, unseen states, the two pre-registered tests |

### Pick a mode (`MODE` in Part 1)
* `view` — trains nothing; prints the stored results.
* `quick` — retrains **XGBoost and the ANN** (one seed) and compares with the published numbers. Writes to `results_quick/`; nothing else is touched.
* `full` — reproduces everything (7 models × 2 variants × 3 seeds, unseen-state tests, pre-registered tests, tables). About 10 CPU-hours: split it with `STEPS` and use *Save & Run All* on Kaggle.

### Protocol (identical for every model)
Train 2000-2015, choose settings on 2016-2017, test on 2018-2022 (40,630 rows). Model A = no rainfall, Model B = with rainfall. Target = capped anomaly z of a **hidden** historical reading; stress class from the predicted z (1 / 1.5 / 2). 3 seeds. The test split is never used for any choice.


## Part 1 · Settings and project files

In [ ]:
MODE = "view"          # "view" | "quick" | "full"
STEPS = "chrono,geo,robust,evaluate"   # full mode only; on Kaggle (12 h limit) run them in separate sessions

import os, sys, glob, zipfile, subprocess, time, shutil
from pathlib import Path
BUNDLE_ZIP = None      # e.g. "/content/bde_bundle_core.zip" if auto-detect fails
GITHUB_URL = "https://github.com/dhruvvvgg/bde-team7.git"   # only works after the project was pushed

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = (not ON_KAGGLE) and Path("/content").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()

def looks_ok(p):
    p = Path(p); return (p / "modeling" / "data.py").exists() and (p / "processed" / "gw_features_by_state").exists()

def writable(c):
    """Kaggle mounts datasets read-only; copy the project to the writable working folder so quick/full mode can write results."""
    c = Path(c)
    if str(c).startswith("/kaggle/input"):
        dest = WORK / "bde-team7"
        if not looks_ok(dest):
            print("copying the project from the read-only dataset to", dest, "...")
            shutil.copytree(c, dest, dirs_exist_ok=True)
        return dest
    return c

def find_root():
    cands = [os.environ.get("BDE_ROOT"), Path.cwd(), Path.cwd().parent, WORK / "bde-team7"]
    cands += [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/modeling/data.py", recursive=True)]
    for c in cands:
        if c and looks_ok(c): return writable(c)
    zips = ([BUNDLE_ZIP] if BUNDLE_ZIP else []) + glob.glob("/kaggle/input/**/bde_bundle_core*.zip", recursive=True) + glob.glob("/content/**/bde_bundle_core*.zip", recursive=True) + glob.glob(str(Path.cwd() / "bde_bundle_core*.zip"))
    for z in zips:
        if z and Path(z).exists():
            print("unzipping", Path(z).name); zipfile.ZipFile(z).extractall(WORK)
            for c in (WORK / "bde-team7", WORK):
                if looks_ok(c): return c
    dest = WORK / "bde-team7"
    if not dest.exists():
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "review2-modeling", GITHUB_URL, str(dest)])   # branch until merged into main
        if r.returncode: subprocess.run(["git", "clone", "-q", "--depth", "1", GITHUB_URL, str(dest)])
    if looks_ok(dest): return dest
    raise SystemExit("Project files not found: upload bde_bundle_core.zip (Colab: Files panel; Kaggle: Add Data) and rerun.")

ROOT = find_root(); os.environ["BDE_ROOT"] = str(ROOT); os.chdir(ROOT)
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local machine", "| project folder:", ROOT.name, "| cpus:", os.cpu_count())

In [ ]:
import importlib
miss = [p for m, p in {"xgboost": "xgboost", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "scipy": "scipy"}.items() if importlib.util.find_spec(m) is None]
if miss: subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + miss)
import torch                                   # import torch BEFORE pandas (matters on Windows; harmless elsewhere)
import numpy as np, pandas as pd
sys.path.insert(0, str(ROOT / "modeling")); sys.path.insert(0, str(ROOT / "src"))
print("torch", torch.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## Part 2 · Check the data
`processed/` is the committed output of the preprocessing pipeline (pandas reference, reproduced by the Spark and BigQuery versions). 60 of 84 columns are safe model inputs (class A), 13 are descriptive only (B), 11 are labels (C), see `docs/handoff_for_modeling.md`.

In [ ]:
import data as D
b = D.build()
m = b["meta"]
print("model rows (eval-eligible):", len(m), "| inputs A / B:", len(b["names"]) - b["is_rain"].sum(), "/", len(b["names"]), "| sequence tensor:", b["seq"].shape)
display(pd.crosstab(m.split_chrono, m.season).loc[["train", "val", "test"]])
print("forbidden location/year columns among the inputs:", sorted({"lat", "lon", "state", "district", "year"} & set(b["names"])) or "none")

## Part 3 · Code map
Every script of the modeling study with the first line of its docstring, and the order `run_pipeline.py --mode full` runs them.

In [ ]:
import ast
files = sorted((ROOT / "modeling").glob("*.py"))
rows = []
for f in files:
    doc = (ast.get_docstring(ast.parse(f.read_text(encoding="utf-8"))) or "").strip().split("\n")[0]
    rows.append([f.name, len(f.read_text(encoding="utf-8").splitlines()), doc[:110]])
display(pd.DataFrame(rows, columns=["file in modeling/", "lines", "purpose (first docstring line)"]))

In [ ]:
import inspect, importlib.util
spec = importlib.util.spec_from_file_location("run_pipeline", ROOT / "modeling" / "run_pipeline.py"); rp = importlib.util.module_from_spec(spec); spec.loader.exec_module(rp)
print("Pipeline order (full mode):")
for name in ("step_chrono", "step_geo", "step_robust", "step_evaluate"):
    print(f"\n--- {name} ---"); print(inspect.getsource(getattr(rp, name)))

In [ ]:
import lib as Lb
print("Hyperparameter grids (3-4 configurations per model, chosen on VALIDATION only):")
for k, v in Lb.GRIDS.items():
    print(f"  {k:8s} {len(v)} configs, e.g. {v[0]}")

## Part 4 · Run the models
* **view** does nothing here.
* **quick** retrains XGBoost and ANN (one seed). Expected: XGBoost A 1.264 / B 1.120 m; ANN seed 0 A 1.300 / B 1.142 m test depth MAE (the report's 3-seed ANN means are 1.270 / 1.125).
* **full** runs `modeling/run_pipeline.py --mode full --steps STEPS`; logs go to `results/logs/` and the run **overwrites** `results/`.

In [ ]:
t0 = time.time()
if MODE == "quick":
    r = subprocess.run([sys.executable, "-u", "modeling/run_pipeline.py", "--mode", "quick"], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-3500:]); print(r.stderr[-1200:])
    print("quick run finished in %.0f s (exit code %d)" % (time.time() - t0, r.returncode))
elif MODE == "full":
    log = open(ROOT / "full_pipeline.log", "w")
    p = subprocess.Popen([sys.executable, "-u", "modeling/run_pipeline.py", "--mode", "full", "--steps", STEPS], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)
    print("full pipeline started in the background (pid %d); monitor it with the next cell." % p.pid)
else:
    print("MODE = view: nothing to train. The stored results below come from the full study.")

In [ ]:
if MODE == "full":
    print(subprocess.run(["tail", "-n", "15", str(ROOT / "full_pipeline.log")], capture_output=True, text=True).stdout)
    for d in ("chrono", "geo", "robust"):
        print(d, len(list((ROOT / "results" / d).glob("*"))), "files")
else:
    print("(monitor cell: only used in full mode)")

## Part 5 · Result tables
Every number in the report, paper and slides comes from `results/tables/`.

In [ ]:
T = ROOT / "results" / "tables"
s = pd.read_csv(T / "chrono_summary.csv")
print("Main comparison, test split 2018-2022 (mean and sd over 3 seeds; A = no rainfall, B = with rainfall):")
display(s[["model", "variant", "depth_MAE_mean", "depth_MAE_std", "anom_MAE_mean", "anom_R2_mean", "macro_F1_mean", "AUC_stressed_mean"]].round(4))

In [ ]:
for name, note in [("bootstrap.csv", "Rainfall effect (B minus A) and TGT-M versus the others, 95% bootstrap CI over wells"),
                   ("geo_summary.csv", "Unseen-state results (g5 = grouped 5-fold, loso = leave-one-state-out)"),
                   ("interval_summary.csv", "Pre-registered H1: 80% intervals"), ("interval_comparison.csv", "H1: TGT-M interval score minus the others (negative = TGT-M better)"),
                   ("robust_summary.csv", "Pre-registered H2: anomaly MAE versus share of recent readings removed"), ("robust_degradation.csv", "H2: degradation and differences with CI"),
                   ("generalization_gap.csv", "Post hoc: unseen minus seen error")]:
    f = T / name
    if f.exists():
        print("\n" + note); display(pd.read_csv(f).round(4))

## Appendix
**Reading the numbers.** *Depth MAE* = mean absolute error of the predicted groundwater depth in metres (lower is better). *Anomaly* = (reading − the well's past same-season mean) / its SD, capped at ±5. *Macro-F1* = mean F1 over the 4 stress classes. *Interval score* = interval width + 10 × the amount by which the truth falls outside the 80% interval (lower is better). TGT-M ranks 5th of 7 learned models on accuracy; rainfall improves every model; H1 and H2a (pre-registered) are supported, H2b is not.

**Troubleshooting.** *Project files not found* → upload `bde_bundle_core.zip` and re-run Part 1. *Out of memory in full mode* → run one `STEPS` value at a time. Neural networks are not bit-for-bit reproducible across machines: differences of about 0.01 m are normal.
